# Digital workflow on a public PMD infrastructure

## Overview
- Datamanagement and semantics
    - User has a semantically annotated data (example: tensile test on S355 steel, 3 complementary datasets via 3 providers)
    - Collaboration: datasets loaded to a single Ontodocker instance (Ontodocker = triplestore(s) + usermanagement (local and/or identity provider))
    - dataset inspection via SPARQL
    - retrieval
- Analisis
    - analisis of tensile data ($E$, $R_{p,0.2}$)
    - generation of figures
- Publication (still mutable) via MaterialDigital-Dataportal
    - Metadata construction
    - Upload of assets
    - Inspection of published dataset:
        - DCAT RDF (SPARQL)
        - semantic assets (SPARQL)
        - further artifacts (e.g. figures) (verification)
        - retrieval

## Involved PMD infrastructure
- Ontodocker
- MD Dataportal
- indirectly: SSO (weblogin for token generation)
- ...

In this notebook, we will access those services via their APIs. To simplify this we use `courier`, which provides Python clients for working with remote services used to publish, upload, query, and manage research and semantic assets. To achieve this, `courier` provides explicit Python APIs and simplifies access and authentication as well as it make further service features more easily accessible.

## Getting started

### Environment
This notebook assumes that these environment variable are available in the execution env (i.e. the one from which the kernel is instantiated):
```
ONTODOCKER_ADDRESS
ONTODOCKER_TOKEN
DATAPORTAL_TOKEN
```
You can do this for example by sourcing an appropriate `.env` file before starting the notebook server.

#### Verify the environment

In [1]:
import os

def require_env(name: str) -> str:
    value = os.getenv(name)
    if value is None or not value.strip():
        raise RuntimeError(f"Required environment variable {name} is not set.")
    return value.strip()

ontodocker_address = require_env("ONTODOCKER_ADDRESS")
ontodocker_token = os.getenv("ONTODOCKER_TOKEN") or None

dataportal_token = require_env("DATAPORTAL_TOKEN")

In [2]:
ontodocker_address # you should avoid printing tokens

'ontodocker.pmds-mpcdf-2.pyiron.com'

### Access Ontodocker via `courier`

In [3]:
from courier import OntodockerClient

ontodocker = OntodockerClient(address=ontodocker_address, token=ontodocker_token)
ontodocker.base_url

'https://ontodocker.pmds-mpcdf-2.pyiron.com'

#### Explore datasets and (SPARQL-)endpoints

In [4]:
ontodocker.datasets.list()

['pmdco2_tto_example_parallel', 'pmdco2_tto_example_perpendicular']

In [5]:
ontodocker.endpoints.list()

[EndpointInfo(dataset='pmdco2_tto_example_parallel', sparql_endpoint='https://ontodocker.pmds-mpcdf-2.pyiron.com/api/v1/jena/pmdco2_tto_example_parallel/sparql'),
 EndpointInfo(dataset='pmdco2_tto_example_perpendicular', sparql_endpoint='https://ontodocker.pmds-mpcdf-2.pyiron.com/api/v1/jena/pmdco2_tto_example_perpendicular/sparql')]

#### Create and populate a new dataset

First, we need to ceate a dataset, which is empty in the beginning.

In [6]:
dataset = "pmdco2_tto_example_diagonal"
ontodocker.datasets.create(dataset)

'"Dataset name pmdco2_tto_example_diagonal created"'

The new dataset will now be populated with RDF data. `OntodockerClient` accepts either `.ttl` serialized graphs or `rdflib.graph` instances. Here, we want to upload a file `../datasets/pmdco2_tto_example_diagonal.ttl`:

In [7]:
from pathlib import Path

turtle_path = Path("pmd_public-infrastructure-demonstrator/datasets/pmdco2_tto_example_diagonal.ttl")
assert turtle_path.exists(), turtle_path
turtle_path

PosixPath('pmd_public-infrastructure-demonstrator/datasets/pmdco2_tto_example_diagonal.ttl')

In [8]:
ontodocker.datasets.upload_turtlefile(dataset, turtle_path)

'"Upload succeeded { \\n  \\"count\\" : 920 ,\\n  \\"tripleCount\\" : 920 ,\\n  \\"quadCount\\" : 0\\n}\\n"'

#### Dataset investigation via `SPARQL`

`pmd_public-infrastructure-demonstrator/query_collection.py` provides helpful SPARQL queries alongside a suitable `dataclass` `SparqlQuery` to wrap them together with their query variables.

In [9]:
import pandas as pd, hashlib
pd.set_option('display.max_colwidth', None)
pd.set_option('display.colheader_justify', 'left')
pd.set_option('display.width', 200)

In [10]:
import sys


query_collection_dir = Path("pmd_public-infrastructure-demonstrator")
sys.path.insert(0, str(query_collection_dir))

from query_collection import TestQueries, S355queries

In [11]:
query = S355queries.material_designation()

In [12]:
print(query.query)


        PREFIX pmd: <https://w3id.org/pmd/co/>
        SELECT DISTINCT (?p AS ?uri) (?matDesVal AS ?materialDesignation)
        WHERE {
            ?s a pmd:TestPiece .
            ?p pmd:input ?s .
            ?p pmd:characteristic ?matDes .
            ?matDes a pmd:materialDesignation .
            ?matDes pmd:value ?matDesVal .
        }
        ORDER BY ?p
        


In [13]:
query.query_vars

['uri', 'materialDesignation']

Datasets can now be queried like this:

In [14]:
result = ontodocker.sparql.query_df(dataset, query.query, query.query_vars)
result

,uri,materialDesignation
0,https://w3id.org/pmd/ao/tte/pmdao-tto-tt-S355-10_process,S355
1,https://w3id.org/pmd/ao/tte/pmdao-tto-tt-S355-9_process,S355


In [15]:
query = S355queries.orientation()
result = ontodocker.sparql.query_df(dataset, query.query, query.query_vars)
result

,uri,cut_orientation
0,https://w3id.org/pmd/ao/tte/pmdao-tto-tt-S355-10_process,diagonal to rolling direction
1,https://w3id.org/pmd/ao/tte/pmdao-tto-tt-S355-9_process,diagonal to rolling direction


In [16]:
query = S355queries.csv_url()
result = ontodocker.sparql.query_df(dataset, query.query, query.query_vars)
result

,uri,url
0,https://w3id.org/pmd/ao/tte/pmdao-tto-tt-S355-10_process,https://data.pmds-mpcdf-2.pyiron.com/Zd3.csv
1,https://w3id.org/pmd/ao/tte/pmdao-tto-tt-S355-9_process,https://data.pmds-mpcdf-2.pyiron.com/Zd2.csv


In [17]:
query = S355queries.csv_columns()
result = ontodocker.sparql.query_df(dataset, query.query, query.query_vars)
result

,uri,url,name,column_number,quantity,unit
0,https://w3id.org/pmd/ao/tte/pmdao-tto-tt-S355-10_process,https://data.pmds-mpcdf-2.pyiron.com/Zd3.csv,b0,1,https://w3id.org/pmd/co/Duration,http://qudt.org/vocab/unit/SEC
1,https://w3id.org/pmd/ao/tte/pmdao-tto-tt-S355-10_process,https://data.pmds-mpcdf-2.pyiron.com/Zd3.csv,b1,2,https://w3id.org/pmd/co/CrossheadSeparation,http://qudt.org/vocab/unit/MilliM
2,https://w3id.org/pmd/ao/tte/pmdao-tto-tt-S355-10_process,https://data.pmds-mpcdf-2.pyiron.com/Zd3.csv,b2,3,https://w3id.org/pmd/co/Force,http://qudt.org/vocab/unit/KiloN
3,https://w3id.org/pmd/ao/tte/pmdao-tto-tt-S355-10_process,https://data.pmds-mpcdf-2.pyiron.com/Zd3.csv,b3,4,https://w3id.org/pmd/co/PercentageExtension,http://qudt.org/vocab/unit/PERCENT
4,https://w3id.org/pmd/ao/tte/pmdao-tto-tt-S355-9_process,https://data.pmds-mpcdf-2.pyiron.com/Zd2.csv,b4,1,https://w3id.org/pmd/co/Duration,http://qudt.org/vocab/unit/SEC
5,https://w3id.org/pmd/ao/tte/pmdao-tto-tt-S355-9_process,https://data.pmds-mpcdf-2.pyiron.com/Zd2.csv,b5,2,https://w3id.org/pmd/co/CrossheadSeparation,http://qudt.org/vocab/unit/MilliM
6,https://w3id.org/pmd/ao/tte/pmdao-tto-tt-S355-9_process,https://data.pmds-mpcdf-2.pyiron.com/Zd2.csv,b6,3,https://w3id.org/pmd/co/Force,http://qudt.org/vocab/unit/KiloN
7,https://w3id.org/pmd/ao/tte/pmdao-tto-tt-S355-9_process,https://data.pmds-mpcdf-2.pyiron.com/Zd2.csv,b7,4,https://w3id.org/pmd/co/PercentageExtension,http://qudt.org/vocab/unit/PERCENT


#### Iterating over SPARQL endpoints

In [18]:
endpoints = ontodocker.endpoints.list()

In [19]:
for ep in endpoints:
    query = S355queries.material_designation()
    dataset = ep.dataset
    result = ontodocker.sparql.query_df(dataset, query.query, query.query_vars)
    print(f"dataset: {dataset}")
    print(f"{result}\n")

dataset: pmdco2_tto_example_parallel
  uri                                                      materialDesignation
0  https://w3id.org/pmd/ao/tte/pmdao-tto-tt-S355-1_process  S355              
1  https://w3id.org/pmd/ao/tte/pmdao-tto-tt-S355-2_process  S355              
2  https://w3id.org/pmd/ao/tte/pmdao-tto-tt-S355-3_process  S355              
3  https://w3id.org/pmd/ao/tte/pmdao-tto-tt-S355-4_process  S355              

dataset: pmdco2_tto_example_perpendicular
  uri                                                      materialDesignation
0  https://w3id.org/pmd/ao/tte/pmdao-tto-tt-S355-5_process  S355              
1  https://w3id.org/pmd/ao/tte/pmdao-tto-tt-S355-6_process  S355              
2  https://w3id.org/pmd/ao/tte/pmdao-tto-tt-S355-7_process  S355              
3  https://w3id.org/pmd/ao/tte/pmdao-tto-tt-S355-8_process  S355              

dataset: pmdco2_tto_example_diagonal
  uri                                                       materialDesignation
0  https://w

#### Download RDF data

In [20]:
for ep in endpoints:
    dataset = ep.dataset
    path = Path("pmd_public-infrastructure-demonstrator") / "downloaded" / f"{dataset}.ttl"
    ontodocker.datasets.download_turtle(dataset, filename=path)

### Generate a RDF artifact for publication
Concatenation of the three datasets we just downloaded using `rdflib`:

In [21]:
from rdflib import Graph

base = Path("pmd_public-infrastructure-demonstrator")

downloaded_dir = base / "downloaded"
output_file = base / "datasets" / "pmdco2_tto_recombined.ttl"

input_files = sorted(downloaded_dir.glob("*.ttl"))

graph = Graph()

for path in input_files:
    graph.parse(path, format="turtle")

graph.serialize(destination=output_file, format="turtle")

<Graph identifier=Ne9737bacd19b4b9baaefc60005c2bae1 (<class 'rdflib.graph.Graph'>)>

### Publication on the MaterialDigital Dataportal

The recombined RDF artifact can now be published as a Dataportal dataset. The TTL file is both the publication asset and a source for publication metadata. We split this into two layers: `PublicationMetadata` contains the human-facing scientific description, while `DataportalMetadata` adds Dataportal-specific package settings and machine-checkable audit extras.

#### Access the Dataportal via `courier`

In [22]:
from courier import DataportalClient

dataportal = DataportalClient(api_token=dataportal_token)
dataportal.base_url

'https://dataportal.material-digital.de'

#### Generate publication metadata from the RDF artifact

The namespace handles below point to RDF vocabularies already used in the Turtle file: Dublin Core (`dc`) for bibliographic fields, CSVW (`csvw`) for CSV resources, RDFS (`rdfs`) for labels, and PMDco (`pmd`) for the tensile-test domain model.

In [23]:
from datetime import date
from pathlib import Path

import rdflib
from courier import Person, PublicationMetadata
from rdflib import Graph

ttl_path = Path("pmd_public-infrastructure-demonstrator/datasets/pmdco2_tto_recombined.ttl")
source_files = sorted(Path("pmd_public-infrastructure-demonstrator/downloaded").glob("*.ttl"))

publication_graph = Graph()
publication_graph.parse(ttl_path, format="turtle")

DC = rdflib.Namespace("http://purl.org/dc/elements/1.1/")
CSVW = rdflib.Namespace("http://www.w3.org/ns/csvw#")
PMD = rdflib.Namespace("https://w3id.org/pmd/co/")
RDFS = rdflib.Namespace("http://www.w3.org/2000/01/rdf-schema#")

In [24]:
creator_orcids = sorted({str(obj) for obj in publication_graph.objects(None, DC.creator)})
creator_labels = sorted(
    {
        str(label)
        for creator in creator_orcids
        for label in publication_graph.objects(rdflib.URIRef(creator), RDFS.label)
    }
)
csv_urls = sorted({str(obj) for obj in publication_graph.objects(None, CSVW.url)})

test_count = len(
    set(publication_graph.subjects(rdflib.RDF.type, PMD.TensileTest))
)

orientation_query = S355queries.orientation()
orientation_index = orientation_query.query_vars.index("cut_orientation")
orientations = sorted(
    {str(row[orientation_index]) for row in publication_graph.query(orientation_query.query)}
)

{
    "triples": len(publication_graph),
    "tensile_tests": test_count,
    "csv_resources": len(csv_urls),
    "orientations": orientations,
    "creator_orcids": creator_orcids,
    "creator_labels": creator_labels,
}

{'triples': 3352,
 'tensile_tests': 10,
 'csv_resources': 10,
 'orientations': ['diagonal to rolling direction',
  'in rolling direction',
  'perpendicular to rolling direction'],
 'creator_orcids': ['https://orcid.org/0000-0001-7192-7143',
  'https://orcid.org/0000-0002-3717-7104',
  'https://orcid.org/0000-0002-7094-5371',
  'https://orcid.org/0000-0003-4971-3645'],
 'creator_labels': ['Bernd Bayerlein',
  'Jörg Waitelonis',
  'Markus Schilling',
  'Philipp von Hartrott']}

In [25]:
creators = tuple(Person(name=name) for name in creator_labels)
if not creators:
    creators = tuple(
        Person(name=orcid, orcid=orcid.rsplit("/", 1)[-1])
        for orcid in creator_orcids
    )

metadata = PublicationMetadata(
    title="Recombined S355 tensile test RDF dataset",
    description=(
        "Recombined RDF/Turtle artifact generated from three distributed "
        "Ontodocker datasets for the PMD public infrastructure demonstrator. "
        f"The graph contains {test_count} tensile tests and references "
        f"{len(csv_urls)} CSV resources. Specimen orientations: "
        f"{', '.join(orientations)}."
    ),
    creators=creators,
    publication_date=date.today(),
    keywords=(
        "Platform MaterialDigital",
        "PMD",
        "S355",
        "tensile test",
        "RDF",
        "Turtle",
        "CSVW",
        "Ontodocker",
    ),
    license="CC-BY-4.0",
    #language="en",
)

metadata

PublicationMetadata(title='Recombined S355 tensile test RDF dataset', description='Recombined RDF/Turtle artifact generated from three distributed Ontodocker datasets for the PMD public infrastructure demonstrator. The graph contains 10 tensile tests and references 10 CSV resources. Specimen orientations: diagonal to rolling direction, in rolling direction, perpendicular to rolling direction.', creators=(Person(family_name=None, given_names=None, name='Bernd Bayerlein', affiliation=None, orcid=None, gnd=None), Person(family_name=None, given_names=None, name='Jörg Waitelonis', affiliation=None, orcid=None, gnd=None), Person(family_name=None, given_names=None, name='Markus Schilling', affiliation=None, orcid=None, gnd=None), Person(family_name=None, given_names=None, name='Philipp von Hartrott', affiliation=None, orcid=None, gnd=None)), publication_date=datetime.date(2026, 7, 20), contributors=(), keywords=('Platform MaterialDigital', 'PMD', 'S355', 'tensile test', 'RDF', 'Turtle', 'CSVW

#### Select the owning Dataportal organization

Dataportal datasets must belong to an organization. We ask CKAN which organizations the current token may create datasets in and use the organization `name` as `owner_org`.

In [26]:
organizations = dataportal.action.call(
    "organization_list_for_user",
    {"permission": "create_dataset"},
)

organization_options = [
    {
        "name": organization.get("name"),
        "title": organization.get("title"),
        "id": organization.get("id"),
    }
    for organization in organizations
]

organization_options

[{'name': 'mpi-susmat',
  'title': 'MPI-SusMat',
  'id': 'd9fe78b4-187e-43ae-80c1-5ac7d20557ad'},
 {'name': 'pmd', 'title': 'PMD', 'id': '4db1a5d1-c990-406d-9dbf-a0ca384481d0'}]

In [27]:
if not organization_options:
    raise RuntimeError(
        "No Dataportal organization with dataset creation permission was found "
        "for the configured token."
    )

owner_org = organization_options[0]["name"]
owner_org

'mpi-susmat'

#### Add Dataportal audit metadata

The `extras` fields make the Dataportal record auditable: the checksum identifies the exact uploaded bytes, while graph size, namespaces, and RDF type counts summarize the semantic artifact without replacing the file itself.

In [28]:
import hashlib
import json
from collections import Counter

from courier.services.dataportal.metadata import DataportalMetadata

ttl_bytes = ttl_path.read_bytes()
sha256 = hashlib.sha256(ttl_bytes).hexdigest()

namespaces = {
    prefix: str(namespace)
    for prefix, namespace in publication_graph.namespaces()
    if prefix
}

type_counts = Counter(
    str(obj) for obj in publication_graph.objects(None, rdflib.RDF.type)
)

audit_extras = {
    "source_file": str(ttl_path),
    "source_files": json.dumps(
        [str(path) for path in source_files],
        sort_keys=True,
        separators=(",", ":"),
    ),
    "sha256": sha256,
    "file_size_bytes": str(ttl_path.stat().st_size),
    "rdf_triple_count": str(len(publication_graph)),
    "rdf_namespaces": json.dumps(
        namespaces,
        sort_keys=True,
        separators=(",", ":"),
    ),
    "rdf_type_counts": json.dumps(
        dict(sorted(type_counts.items())),
        sort_keys=True,
        separators=(",", ":"),
    ),
    "creator_orcids": json.dumps(
        creator_orcids,
        sort_keys=True,
        separators=(",", ":"),
    ),
    "csv_resource_urls": json.dumps(
        csv_urls,
        sort_keys=True,
        separators=(",", ":"),
    ),
    "pmd_profile": "pmdco2-tto-recombined",
}

dataportal_metadata = DataportalMetadata(
    metadata=metadata,
    name="pmdco2-tto-recombined",
    owner_org=owner_org,
    private=False,
    extras=audit_extras,
)

dataportal_metadata.to_payload()

{'name': 'pmdco2-tto-recombined',
 'title': 'Recombined S355 tensile test RDF dataset',
 'notes': 'Recombined RDF/Turtle artifact generated from three distributed Ontodocker datasets for the PMD public infrastructure demonstrator. The graph contains 10 tensile tests and references 10 CSV resources. Specimen orientations: diagonal to rolling direction, in rolling direction, perpendicular to rolling direction.',
 'tags': [{'name': 'Platform MaterialDigital'},
  {'name': 'PMD'},
  {'name': 'S355'},
  {'name': 'tensile test'},
  {'name': 'RDF'},
  {'name': 'Turtle'},
  {'name': 'CSVW'},
  {'name': 'Ontodocker'}],
 'owner_org': 'mpi-susmat',
 'private': False,
 'license_id': 'CC-BY-4.0',
 'extras': [{'key': 'source_file',
   'value': 'pmd_public-infrastructure-demonstrator/datasets/pmdco2_tto_recombined.ttl'},
  {'key': 'source_files',
   'value': '["pmd_public-infrastructure-demonstrator/downloaded/pmdco2_tto_example_diagonal.ttl","pmd_public-infrastructure-demonstrator/downloaded/pmdco2_t

#### Publish the dataset and assets

Executing this cell creates the Dataportal record, uploads the recombined Turtle file, and links the CSV resources referenced by the RDF graph.

In [29]:
from courier.services.ckan.exceptions import CkanApiError

try:
    dataset = dataportal.datasets.create(dataportal_metadata)
except CkanApiError as exc:
    print(json.dumps(exc.payload, indent=2))
    raise

ttl_asset = dataportal.assets.upload_rdf(
    dataset,
    ttl_path,
    name=ttl_path.name,
    description="Recombined RDF/Turtle metadata graph.",
)

csv_assets = [
    dataportal.assets.create_url(
        dataset,
        url=url,
        name=Path(url).name,
        description="CSV resource referenced by the RDF graph.",
        format="csv",
    )
    for url in csv_urls
]

publication_result = {
    "dataset": dataset,
    "ttl_asset": ttl_asset,
    "csv_assets": csv_assets,
}

publication_result

{'dataset': DataportalDatasetInfo(id='bfbda699-bed1-4c13-afb7-60c0f9f0d295', name='pmdco2-tto-recombined', title='Recombined S355 tensile test RDF dataset', notes='Recombined RDF/Turtle artifact generated from three distributed Ontodocker datasets for the PMD public infrastructure demonstrator. The graph contains 10 tensile tests and references 10 CSV resources. Specimen orientations: diagonal to rolling direction, in rolling direction, perpendicular to rolling direction.', owner_org='d9fe78b4-187e-43ae-80c1-5ac7d20557ad', private=False, dataset_type='dataset', raw={'author': None, 'author_email': None, 'creator_user_id': '80c66f6a-727c-4c26-acf5-80194e3556ef', 'id': 'bfbda699-bed1-4c13-afb7-60c0f9f0d295', 'isopen': False, 'license_id': 'CC-BY-4.0', 'license_title': 'CC-BY-4.0', 'maintainer': None, 'maintainer_email': None, 'metadata_created': '2026-07-20T20:27:29.625873', 'metadata_modified': '2026-07-20T20:27:29.625877', 'name': 'pmdco2-tto-recombined', 'notes': 'Recombined RDF/Turtl

#### Inspect RDF and SPARQL access points

After publication there are two RDF layers worth separating. The uploaded `pmdco2_tto_recombined.ttl` asset is the domain graph for the tensile tests; once the Dataportal has created a SPARQL resource for it, courier can query that endpoint directly. The Dataportal dataset record itself is also exposed as DCAT RDF. That DCAT graph contains bibliographic and catalog metadata such as title, description, organization, tags, and resource links. The CSVW-derived Turtle files generated by the portal may also be queryable; here we only count and list them instead of querying every generated resource.

In [30]:
published_dataset = dataportal.datasets.show(dataset.name)
resources = published_dataset.raw.get("resources", [])

def is_turtle_resource(resource):
    format_value = str(resource.get("format", "")).lower()
    mimetype = str(resource.get("mimetype", "")).lower()
    return "turtle" in format_value or mimetype == "text/turtle"

def is_sparql_resource(resource):
    return str(resource.get("format", "")).strip().lower() == "sparql"

uploaded_ttl_resource = next(
    resource
    for resource in resources
    if resource.get("name") == ttl_path.name
)
sparql_resources = [resource for resource in resources if is_sparql_resource(resource)]
generated_turtle_resources = [
    resource
    for resource in resources
    if is_turtle_resource(resource) and resource.get("wasGeneratedBy")
]
dcat_metadata_urls = {
    "ttl": dataportal.rdf.dataset_url(published_dataset, format="ttl"),
    "jsonld": dataportal.rdf.dataset_url(published_dataset, format="jsonld"),
}

{
    "dataset": published_dataset.name,
    "uploaded_ttl_asset": {
        "name": uploaded_ttl_resource.get("name"),
        "url": uploaded_ttl_resource.get("url"),
    },
    "sparql_endpoints": [
        {"name": resource.get("name"), "url": resource.get("url")}
        for resource in sparql_resources
    ],
    "dcat_metadata_urls": dcat_metadata_urls,
    "generated_turtle_resources": [
        resource.get("name") for resource in generated_turtle_resources
    ],
}


{'dataset': 'pmdco2-tto-recombined',
 'uploaded_ttl_asset': {'name': 'pmdco2_tto_recombined.ttl',
  'url': 'https://dataportal.material-digital.de/dataset/bfbda699-bed1-4c13-afb7-60c0f9f0d295/resource/114a4002-771d-482e-b5d7-2612b9bd28ec/download/pmdco2_tto_recombined.ttl'},
 'sparql_endpoints': [],
 'dcat_metadata_urls': {'ttl': 'https://dataportal.material-digital.de/dataset/bfbda699-bed1-4c13-afb7-60c0f9f0d295.ttl',
  'jsonld': 'https://dataportal.material-digital.de/dataset/bfbda699-bed1-4c13-afb7-60c0f9f0d295.jsonld'},
 'generated_turtle_resources': ['Zd2.ttl',
  'Zd3.ttl',
  'Zx1.ttl',
  'Zx2.ttl',
  'Zx3.ttl',
  'Zx4.ttl',
  'Zy1.ttl',
  'Zy2.ttl',
  'Zy3.ttl',
  'Zy4.ttl']}

The SPARQL endpoint below belongs to the published RDF graph and is queried remotely. The DCAT metadata is retrieved through courier's RDF helper because CKAN/DCAT exposes dataset metadata as RDF serializations; it is the catalog record about the publication, not the tensile-test graph itself.

In [33]:
uploaded_graph_sparql_endpoint = next(
    (resource.get("url") for resource in sparql_resources if resource.get("url")),
    None,
)
if uploaded_graph_sparql_endpoint is None:
    raise RuntimeError(
        "No SPARQL resource is listed for this dataset yet. "
        "Re-run this cell after the Dataportal has generated the endpoint."
    )

triple_count_query = "SELECT (COUNT(*) AS ?triples) WHERE { ?s ?p ?o }"
uploaded_graph_count = dataportal.sparql.query_df(
    uploaded_graph_sparql_endpoint,
    triple_count_query,
    ["triples"],
)

dcat_metadata_graph = Graph()
dcat_metadata_graph.parse(
    data=dataportal.rdf.dataset(published_dataset, format="ttl"),
    format="turtle",
)
dcterms_title = rdflib.URIRef("http://purl.org/dc/terms/title")
dcat_titles = sorted(str(title) for title in dcat_metadata_graph.objects(None, dcterms_title))

{
    "uploaded_graph_sparql_endpoint": uploaded_graph_sparql_endpoint,
    "uploaded_graph_triple_count": uploaded_graph_count,
    "dcat_metadata_ttl_url": dcat_metadata_urls["ttl"],
    "dcat_metadata_triple_count": len(dcat_metadata_graph),
    "dcat_titles": dcat_titles,
}


RuntimeError: No SPARQL resource is listed for this dataset yet. Re-run this cell after the Dataportal has generated the endpoint.

## Cleanup

In [ ]:
#ontodocker.datasets.delete(dataset)

In [ ]:
#dataportal.datasets.delete("pmdco2-tto-recombined")